In [ ]:
import numpy as np
from scipy.special import jv
epsilon_p = .1
E_J=37 * 2 * np.pi
phi_a=0.11
phi_b=0.204
alpha_sq = 8
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 5 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.005 * kappa_2
eps_d = 2 * alpha_sq * g2

In [ ]:
from scipy.optimize import fsolve

def equations(y):
    A, B, PA, PB = y
    eq1 = -np.sqrt(2)*g*(A*B - PA*PB) - kappa_a*(A/2 + PA)
    eq2 =  -eps_d/np.sqrt(2) + g*(A**2 + PA**2)/np.sqrt(2) - kappa_b*(B/2 + PB)
    eq3 = -np.sqrt(2)*g*(A*PB - PA*B) + kappa_a*PA/2
    eq4 =  np.sqrt(2)*g*A*PA + kappa_b*PB/2
    return [eq1, eq2, eq3, eq4]

A_0 = np.sqrt(eps_d/g)
delta = kappa_a*kappa_b/(4*g**2)
a = (A_0**2)/(2*A_0**2 - 2*delta)*(1 + (2*delta)/(A_0**2)-np.sqrt(1+4*(2*delta)/(A_0**2)))
P_A_value = np.sqrt(delta/(a*(a**2-1)))
A_value = a*P_A_value
B_value = -2*np.sqrt(2)*g/kappa_b*A_value**2-kappa_a/(2*np.sqrt(2)*g)
P_B_value = -2*np.sqrt(2)*g*A_value*P_A_value/kappa_b
y0 = [A_value, B_value, P_A_value, P_B_value]  # initial guess
sol = fsolve(equations, y0, full_output=True)

A, B, PA, PB = sol[0]
print(f"A={A:.10f}, B={B:.10f}, PA={PA:.10f}, PB={PB:.10f}")
print(f"Residual: {np.max(np.abs(equations(sol[0]))):.2e}")
print("A_value: "+str(A_value))
print("B_value: "+str(B_value))
print("P_A_value: "+str(P_A_value))
print("P_B_value: "+str(P_B_value))

In [ ]:
kappa_a_list = np.logspace(-4, 0, 100) * kappa_2
delta_list = kappa_a_list*kappa_b/(4*g**2)
ratio_list = 2*delta_list/A_0**2
a_list = 1/(2 - ratio_list)*(1 + ratio_list-np.sqrt(1+4*ratio_list))
PA_sol_list = np.sqrt(delta_list/(a_list*(a_list**2-1)))
A_sol_list = a_list*PA_sol_list
B_sol_list = -2*np.sqrt(2)*g/kappa_b*A_sol_list**2-kappa_a_list/(2*np.sqrt(2)*g)
PB_sol_list = -2*np.sqrt(2)*g*A_sol_list*PA_sol_list/kappa_b

A_fp_1_list = np.sqrt(A_0-delta_list)
B_fp_1_list = -kappa_a_list/(2*np.sqrt(2)*g)

fp_2 = [0, -np.sqrt(2)*eps_d/kappa_b, 0, 0]
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,2)
ax_1, ax_2 = axes

ax_1.scatter(A_sol_list,B_sol_list, c=kappa_a_list)
ax_1.scatter(fp_2[0],fp_2[1])
ax_1.scatter(A_fp_1_list,B_fp_1_list)
ax_1.set_xlabel(r'$A$')
ax_1.set_ylabel(r'$B$')

ax_2.scatter(PA_sol_list,PB_sol_list, c=kappa_a_list)
ax_2.scatter(fp_2[2],fp_2[3])
ax_2.scatter(0,0)
ax_2.set_xlabel(r'$P_A$')
ax_2.set_ylabel(r'$P_B$')

plt.tight_layout()
plt.show()

# Previous numerical study

In [ ]:
kappa_a_list = np.linspace(1e-3, 1e1, 100) * kappa_2
A_sol_list = []
B_sol_list = []
PA_sol_list = []
PB_sol_list = []
for (idx_kappa,kappa_a) in enumerate(kappa_a_list):
    def equations(y):
        A, B, PA, PB = y
        eq1 = -np.sqrt(2)*g*(A*B - PA*PB) - kappa_a*(A/2 + PA)
        eq2 =  -eps_d/np.sqrt(2) + g*(A**2 + PA**2)/np.sqrt(2) - kappa_b*(B/2 + PB)
        eq3 = np.sqrt(2)*g*(A*PB - PA*B) + kappa_a*PA/2
        eq4 =  np.sqrt(2)*g*A*PA + kappa_b*PB/2
        return [eq1, eq2, eq3, eq4]

    if idx_kappa == 0:
        A_0 = np.sqrt(eps_d/g)
        y0 = [0.0, 1.0, A_0, 0.0]  # initial guess
    else : 
        y0 = [A_sol_list[-1], B_sol_list[-1], PA_sol_list[-1], PB_sol_list[-1]]
    sol = fsolve(equations, y0, full_output=True)

    A, B, PA, PB = sol[0]
    A_sol_list.append(A)
    B_sol_list.append(B)
    PA_sol_list.append(PA)
    PB_sol_list.append(PB)
A_sol_list = np.array(A_sol_list)
B_sol_list = np.array(B_sol_list)
PA_sol_list = np.array(PA_sol_list)
PB_sol_list = np.array(PB_sol_list)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(kappa_a_list, A_sol_list)
plt.plot(kappa_a_list, B_sol_list)
plt.plot(kappa_a_list, PA_sol_list)
plt.plot(kappa_a_list, PB_sol_list)
plt.show()

In [ ]:
from scipy.optimize import curve_fit

def inv_kappa_a(kappa_a, a, b,c):
    return a * kappa_a**c + b

fig, axes = plt.subplots(1, 4, figsize=(14, 3))

for ax, sol, name in zip(axes,
                          [A_sol_list, B_sol_list, PA_sol_list, PB_sol_list],
                          ['A', 'B', 'P_A', 'P_B']):
    popt, _ = curve_fit(inv_kappa_a, kappa_a_list, sol)
    ax.plot(kappa_a_list, sol, label='numerical')
    ax.plot(kappa_a_list, inv_kappa_a(kappa_a_list, *popt), '--', label=f'{popt[0]:.2f}*kappa_a^{popt[2]:.2f} + {popt[1]:.2f}')
    ax.set_xlabel('kappa_a')
    ax.set_title(f'${name}$')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
epsilon_p = .1
E_J=37 * 2 * np.pi
phi_a=0.11
phi_b=0.204
alpha_sq = 8
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 5 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.005 * kappa_2
eps_d = 2 * alpha_sq * g2

g_list = np.linspace(.1, 1e0, 50)
A_sol_list = []
B_sol_list = []
PA_sol_list = []
PB_sol_list = []
for (idx_g,g) in enumerate(g_list):
    def equations(y):
        A, B, PA, PB = y
        eq1 = -np.sqrt(2)*g*(A*B - PA*PB) - kappa_a*(A/2 + PA)
        eq2 =  -eps_d/np.sqrt(2) + g*(A**2 + PA**2)/np.sqrt(2) - kappa_b*(B/2 + PB)
        eq3 = np.sqrt(2)*g*(A*PB - PA*B) + kappa_a*PA/2
        eq4 =  np.sqrt(2)*g*A*PA + kappa_b*PB/2
        return [eq1, eq2, eq3, eq4]

    if idx_g == 0:
        A_0 = np.sqrt(eps_d/g)
        y0 = [0.0, 1.0, A_0, 0.0]  # initial guess
    else : 
        y0 = [A_sol_list[-1], B_sol_list[-1], PA_sol_list[-1], PB_sol_list[-1]]
    sol = fsolve(equations, y0, full_output=True)

    A, B, PA, PB = sol[0]
    A_sol_list.append(A)
    B_sol_list.append(B)
    PA_sol_list.append(PA)
    PB_sol_list.append(PB)
A_sol_list = np.array(A_sol_list)
B_sol_list = np.array(B_sol_list)
PA_sol_list = np.array(PA_sol_list)
PB_sol_list = np.array(PB_sol_list)

In [ ]:
import matplotlib.pyplot as plt
#plt.plot(g_list, A_sol_list)
plt.plot(g_list, .7/np.pow(g_list,.5))
#plt.plot(g_list, B_sol_list)
plt.plot(g_list, PA_sol_list)
#plt.plot(g_list, PB_sol_list)
plt.show()

In [ ]:
from scipy.optimize import curve_fit

def inv_g(g, a, b,c):
    return a / g**c + b

fig, axes = plt.subplots(1, 4, figsize=(14, 3))

for ax, sol, name in zip(axes,
                          [A_sol_list*1e5, B_sol_list, PA_sol_list, PB_sol_list],
                          ['A', 'B', 'P_A', 'P_B']):
    try:
        p0 = [-1e-5, 0, 2]
        popt, _ = curve_fit(inv_g, g_list, sol, p0=p0)
    except Exception as exc:
        p0 = [.7, 0, -.5]
        popt, _ = curve_fit(inv_g, g_list, sol, p0=p0)
    ax.plot(g_list, sol, label='numerical')
    ax.plot(g_list, inv_g(g_list, *popt), '--', label=f'{popt[0]:.2f}/g^{popt[2]:.2f} + {popt[1]:.2f}')
    ax.set_xlabel('g')
    ax.set_title(f'${name}$')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
epsilon_p = .1
E_J=37 * 2 * np.pi
phi_a=0.11
phi_b=0.204
alpha_sq = 8
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 5 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.005 * kappa_2
eps_d = 2 * alpha_sq * g2

kappa_b_list = np.linspace(1e-1, 1e1, 100) /10.4
A_sol_list = []
B_sol_list = []
PA_sol_list = []
PB_sol_list = []
for (idx_kappa,kappa_b) in enumerate(kappa_b_list):
    def equations(y):
        A, B, PA, PB = y
        eq1 = -np.sqrt(2)*g*(A*B - PA*PB) - kappa_a*(A/2 + PA)
        eq2 =  -eps_d/np.sqrt(2) + g*(A**2 + PA**2)/np.sqrt(2) - kappa_b*(B/2 + PB)
        eq3 = np.sqrt(2)*g*(A*PB - PA*B) + kappa_a*PA/2
        eq4 =  np.sqrt(2)*g*A*PA + kappa_b*PB/2
        return [eq1, eq2, eq3, eq4]

    if idx_kappa == 0:
        A_0 = np.sqrt(eps_d/g)
        y0 = [0.0, 1.0, A_0, 0.0]  # initial guess
    else : 
        y0 = [A_sol_list[-1], B_sol_list[-1], PA_sol_list[-1], PB_sol_list[-1]]
    sol = fsolve(equations, y0, full_output=True)

    A, B, PA, PB = sol[0]
    A_sol_list.append(A)
    B_sol_list.append(B)
    PA_sol_list.append(PA)
    PB_sol_list.append(PB)
A_sol_list = np.array(A_sol_list)
B_sol_list = np.array(B_sol_list)
PA_sol_list = np.array(PA_sol_list)
PB_sol_list = np.array(PB_sol_list)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(kappa_b_list, A_sol_list)
plt.plot(kappa_b_list, B_sol_list)
plt.plot(kappa_b_list, PA_sol_list)
plt.plot(kappa_b_list, PB_sol_list)
plt.show()

In [ ]:
from scipy.optimize import curve_fit

def inv_kappa_b(kappa_b, a, b,c):
    return a * kappa_b**c + b

fig, axes = plt.subplots(1, 4, figsize=(14, 3))

for ax, sol, name in zip(axes,
                          [A_sol_list, B_sol_list, PA_sol_list, PB_sol_list],
                          ['A', 'B', 'P_A', 'P_B']):
    popt, _ = curve_fit(inv_kappa_b, kappa_b_list, sol)
    ax.plot(kappa_b_list, sol, label='numerical')
    ax.plot(kappa_b_list, inv_kappa_b(kappa_b_list, *popt), '--', label=f'{popt[0]:.2f}*kappa_b^{popt[2]:.2f} + {popt[1]:.2f}')
    ax.set_xlabel('kappa_b')
    ax.set_title(f'${name}$')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
epsilon_p = .1
E_J=37 * 2 * np.pi
phi_a=0.11
phi_b=0.204
alpha_sq = 8
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 5 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.005 * kappa_2
eps_d = 2 * alpha_sq * g2

eps_d_list = np.linspace(1e-1, 2, 100) * alpha_sq * g2
A_sol_list = []
B_sol_list = []
PA_sol_list = []
PB_sol_list = []
for (idx_eps_d,eps_d) in enumerate(eps_d_list):
    def equations(y):
        A, B, PA, PB = y
        eq1 = -np.sqrt(2)*g*(A*B - PA*PB) - kappa_a*(A/2 + PA)
        eq2 =  -eps_d/np.sqrt(2) + g*(A**2 + PA**2)/np.sqrt(2) - kappa_b*(B/2 + PB)
        eq3 = np.sqrt(2)*g*(A*PB - PA*B) + kappa_a*PA/2
        eq4 =  np.sqrt(2)*g*A*PA + kappa_b*PB/2
        return [eq1, eq2, eq3, eq4]

    if idx_eps_d == 0:
        A_0 = np.sqrt(eps_d/g)
        y0 = [0.0, 1.0, A_0, 0.0]  # initial guess
    else : 
        y0 = [A_sol_list[-1], B_sol_list[-1], PA_sol_list[-1], PB_sol_list[-1]]
    sol = fsolve(equations, y0, full_output=True)

    A, B, PA, PB = sol[0]
    A_sol_list.append(A)
    B_sol_list.append(B)
    PA_sol_list.append(PA)
    PB_sol_list.append(PB)
A_sol_list = np.array(A_sol_list)
B_sol_list = np.array(B_sol_list)
PA_sol_list = np.array(PA_sol_list)
PB_sol_list = np.array(PB_sol_list)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(eps_d_list, A_sol_list)
plt.plot(eps_d_list, -1e-3/np.pow(eps_d_list, .5)-5e-4)
#plt.plot(eps_d_list, B_sol_list)
#plt.plot(eps_d_list, PA_sol_list)
#plt.plot(eps_d_list, np.sqrt(eps_d_list/g))
#plt.plot(eps_d_list, PB_sol_list)
plt.show()

In [ ]:
from scipy.optimize import curve_fit

def inv_eps_d(eps_d, a, b,c):
    return a / eps_d**b + c

fig, axes = plt.subplots(1, 4, figsize=(14, 3))

for ax, sol, name in zip(axes,
                          [A_sol_list, B_sol_list, PA_sol_list, PB_sol_list],
                          ['A', 'B', 'P_A', 'P_B']):
    p0 = [-1e-3, .5, -5e-4]
    popt, _ = curve_fit(inv_eps_d, eps_d_list, sol, p0 = p0)
    ax.plot(eps_d_list, sol, label='numerical')
    ax.plot(eps_d_list, inv_eps_d(eps_d_list, *popt), '--', label=f'{popt[0]:.2f}/eps_d^{popt[1]:.2f} + {popt[2]:.2f}')
    ax.set_xlabel('kappa_b')
    ax.set_title(f'${name}$')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
np.sqrt(eps_d)

In [ ]:
import sympy as sp

A, B_d, PA, PB = sp.symbols('A B_d P_A P_B', real=True)
g, A_0, ka, kb , delta= sp.symbols('g A_0 kappa_a kappa_b delta', positive=True)
delta = sp.symbols('delta', positive=True)
sqrt2 = sp.sqrt(2)

dA  = sqrt2*g*(PA*PB - A*B_d)  - ka*PA
dB_d  = g/sqrt2*(A**2+PA**2 - A_0**2 + delta) - kb/2*B_d - kb*PB
dPA = sqrt2*g*(B_d*PA - A*PB)
dPB = sqrt2*g*A*PA + kb/2*PB

PB_sol = sp.solve(dPB, PB)[0]                          # eliminate P_B
PB_sol

In [ ]:
Bd_sol  = sp.solve((dB_d.subs(PB, PB_sol)), B_d)[0]      # eliminate B (P_A != 0 branch)
Bd_sol

In [ ]:
sp.factor(dA.subs(PB, PB_sol).subs(B_d, Bd_sol))

In [ ]:
sp.factor(dPA.subs(PB, PB_sol).subs(B_d, Bd_sol))

In [ ]:
import sympy as sp

A, B, PA, PB = sp.symbols('A B P_A P_B', real=True)
g, A_0, ka, kb , delta, eps_d, a= sp.symbols('g A_0 kappa_a kappa_b delta varepsilon_d a', positive=True)
delta = sp.symbols('delta', positive=True)
sqrt2 = sp.sqrt(2)

H = -eps_d*PB/sqrt2 - sqrt2*g*A*PA*B + g*(A**2+PA**2)*PB/sqrt2 - ka*(A+PA)*PA/2 - kb*(B+PB)*PB/2
H_M_p = sp.simplify(H.subs(PB, -2*sqrt2*g*A*PA/kb).subs(B, -2*sqrt2*g*A**2/kb-ka/(2*sqrt2*g)).subs(A, a*PA)).subs(PA**2, delta/(a*(a**2-1)))
expr = 2*a*kb*(a**2-1)*(a-1)/delta*H_M_p
expr = sp.simplify(expr.subs(delta, ka*kb/4/g**2))

a_eq = sp.solve(expr, a)[0].subs(ka*kb, delta).subs(g*eps_d, eps_d/(4*g)).subs(eps_d/g, A_0**2).subs(A_0**2,1).subs(delta, delta/A_0**2)
a_eq